In [1]:
from supabase import create_client, Client
import os
from dotenv import load_dotenv
import random
import os
import psycopg

load_dotenv(override=True)

True

In [2]:
DATABASE_URL = os.environ["DATABASE_LOGS_URL"]

In [3]:
with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        # Ambil semua table di schema public
        cur.execute("""
            SELECT tablename
            FROM pg_tables
            WHERE schemaname = 'public';
        """)

        tables = cur.fetchall()

        # Drop semua table
        for table in tables:
            table_name = table[0]
            cur.execute(
                f'DROP TABLE IF EXISTS "{table_name}" CASCADE;'
            )

        conn.commit()

In [4]:
with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT tablename
            FROM pg_tables
            WHERE schemaname = 'public';
        """)

        tables = cur.fetchall()

if tables:
    print("Table yang ada:")
    for table in tables:
        print("-", table[0])
else:
    print("Tidak ada table di schema public.")

Tidak ada table di schema public.


### Create ML Flow

In [5]:
import mlflow

In [6]:
# set nama experiment
mlflow.set_experiment("my-experiment")

# start run
with mlflow.start_run():

    # log parameter
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("batch_size", 32)
    mlflow.log_param("optimizer", "adam")

    # log metric
    mlflow.log_metric("accuracy", 0.92)
    mlflow.log_metric("loss", 0.15)

print("Done")

2026/05/24 19:23:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/24 19:23:12 INFO mlflow.store.db.utils: Updating database tables
2026/05/24 19:23:14 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment' does not exist. Creating a new experiment.


Done


### Using Supabase

In [7]:
tracking_uri = os.getenv("DATABASE_LOGS_URL")

In [8]:
mlflow.set_tracking_uri(tracking_uri)

In [9]:
# Set Experiment
experiment_name = "train_ml"

try:
    experiment_id = mlflow.create_experiment(
        experiment_name,
        # artifact_location=artifact_uri
    )
except mlflow.exceptions.MlflowException:
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(
    experiment_name
)

2026/05/24 19:27:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/24 19:27:37 INFO mlflow.store.db.utils: Updating database tables


<Experiment: artifact_location='/home/ridwanfatur/work/portfolio/modular-ai/vector-databases/supabase/mlruns/1', creation_time=1779625688066, experiment_id='1', last_update_time=1779625688066, lifecycle_stage='active', name='train_ml', tags={}, trace_location=None, workspace='default'>

In [10]:
with mlflow.start_run():

    # log parameter
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("batch_size", 32)
    mlflow.log_param("optimizer", "adam")

    # log metric
    mlflow.log_metric("accuracy", 0.92)
    mlflow.log_metric("loss", 0.15)

### Logs Artifact

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from safetensors.torch import save_file

# Model
model = nn.Linear(1, 1)

In [12]:
# Dataset
X = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0]])
y = torch.tensor([[3.0], [5.0], [7.0], [9.0], [11.0]])

# Optimizer and Loss Function
criterion = nn.MSELoss()

learning_rate = 0.01
epochs = 10

optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [13]:
# Start experiment
with mlflow.start_run():

    # Log parameters
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("epochs", epochs)

    for epoch in range(epochs):
        # Forward pass
        predictions = model(X)
        loss = criterion(predictions, y)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Simple accuracy calculation
        with torch.no_grad():
            mae = torch.mean(torch.abs(predictions - y))
            accuracy = 1 / (1 + mae.item())

        # Log metrics
        mlflow.log_metric("loss", loss.item(), step=epoch)
        mlflow.log_metric("accuracy", accuracy, step=epoch)

        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}")

    # Save model as artifact
    model_path = "model.safetensors"
    save_file(model.state_dict(), model_path)
    mlflow.log_artifact(model_path)

    # Remove local model file after logging
    os.remove(model_path)

Epoch 0, Loss: 45.2389, Accuracy: 0.1371
